The following operations are performaed in silver tables :-
- Schema enforcement / column schema evolution 
- Handling of null and missing values- 
-  Data deduplication- 
-  Resolution of out-of-order and late-arriving data issues- 
-  Data quality checks and enforcement- 
-  Schema evolution- 
-  Type casting / data type handling
- case conversion 
- Trimming 
- maintain order of the data 
- JOINS




Start modeling data:- 

It is common to start performing data modeling in the silver layer, including choosing how to represent heavily nested or semi-structured data:
- Use VARIANT data type.
- Use JSON strings.
- Create structs, maps, and arrays.
- Flatten schema or normalize data into multiple tables.

In [0]:
%sql
describe table intellibi_catalog.intellibi_silver.cleanedCustomer

In [0]:
dbutils.widgets.dropdown(
    name="Environment",
    defaultValue="dev",
    choices=["dev","qa","prd"],
    label="Select Environment"
)

In [0]:
env=dbutils.widgets.get("Environment")

catalog = f"saleslake_{env}"
schema = f"silver_{env}"
table_name = f"{catalog}.{schema}.cleanedCustomer"
print(table_name)


In [0]:

sql_query=f"""
insert into {table_name}  (
select distinct
cast(trim(customer_id) as int ) as customer_id,
trim(customer_name) ,
trim(email) ,
trim(phone)  ,
trim(address) ,
trim(city) ,
trim(state) ,
trim(country),
trim(zip_code ),
trim(segment ),
current_timestamp() as ingest_ts 
from saleslake_{env}.bronze_{env}.rawCustomer
where saleslake_{env}.bronze_{env}.rawCustomer.ingest_ts > 
                    (select coalesce( max(ingest_ts), to_date('1990-01-01','yyyy-MM-dd') ) from {table_name})
                    
order by cast(trim(customer_id) as int)
)"""

spark.sql(sql_query).show()



In [0]:
%sql
select * from saleslake_dev.silver_dev.cleanedCustomer